In [1]:
import pandas as pd
import numpy as np
import json
# from scipy.spatial.distance import cosine

In [2]:
tender_order = [
    "Measles",
    "Mumps",
    "Rubella",
    'MMR',
    "Diphtheria",
    "Tetanus",
    "Pertussis",
    "Hib",
    "Polio",
    "Hepatitis_B",
    "Penta",
    "HPV",
    "PCV",
    "Rotavirus",
]

In [32]:
# MMR and Penta not as antigens
with open("MMR_Penta_S2_D2_overlap_cap.json") as json_file:
    F_results = json.load(json_file)

# start_DF = pd.read_excel("Starting_point.xlsx", sheet_name="F_start")
# start_DF["Value"] = 1
# start_DF.rename(columns={"Antigen": "Tender", "Starting": "Start", "Ending": "Finish"}, inplace=True)

# Search results by keyword for F (tender schedules)
grouped_results = {k: v for k, v in F_results.items() if k.startswith("F")}

F_values_df = pd.DataFrame.from_dict(grouped_results, orient="index").reset_index()
F_values_df.columns = ["index", "Value"]
F_values_df[["F", "Tender", "Start", "Finish"]] = F_values_df["index"].str.extract(r"(\w+)\[(\w+),\((\d+), (\d+)\)\]")
F_values_df = F_values_df.astype({"Start": "int32", "Finish": "int32", "Value": "float64"})

F_values_df.drop(['index', 'F'], axis=1, inplace=True)
F_values_df = F_values_df[['Tender', 'Start', 'Finish', 'Value']]

#offset start value for graphing to be consistent with definition
F_values_df['Finish2'] = F_values_df['Finish']
F_values_df.drop(columns='Finish', inplace=True)
F_values_df.rename(columns={'Finish2': 'Finish'}, inplace=True)

# result_df = pd.concat([F_values_df, start_DF], ignore_index=True)
F_values_df["Tender"] = pd.Categorical(F_values_df["Tender"], categories=tender_order, ordered=True)
F_values_df = F_values_df.sort_values(["Tender","Start", "Finish"])

# df1 = result_df[(result_df.Tender == "Measles") & (result_df.Value == 1)]
df1 = F_values_df

In [33]:
# comparison file
with open("Deterministic_results_F_Q_I_S_base_35.json") as json_file:
    F_results = json.load(json_file)

# start_DF = pd.read_excel("Starting_point.xlsx", sheet_name="F_start")
# start_DF["Value"] = 1
# start_DF.rename(columns={"Antigen": "Tender", "Starting": "Start", "Ending": "Finish"}, inplace=True)

# Search results by keyword for F (tender schedules)
grouped_results = {k: v for k, v in F_results.items() if k.startswith("F")}

F_values_df = pd.DataFrame.from_dict(grouped_results, orient="index").reset_index()
F_values_df.columns = ["index", "Value"]
F_values_df[["F", "Tender", "Start", "Finish"]] = F_values_df["index"].str.extract(r"(\w+)\[(\w+),\((\d+), (\d+)\)\]")
F_values_df = F_values_df.astype({"Start": "int32", "Finish": "int32", "Value": "float64"})

F_values_df.drop(['index', 'F'], axis=1, inplace=True)
F_values_df = F_values_df[['Tender', 'Start', 'Finish', 'Value']]

#offset start value for graphing to be consistent with definition
F_values_df['Finish2'] = F_values_df['Finish']
F_values_df.drop(columns='Finish', inplace=True)
F_values_df.rename(columns={'Finish2': 'Finish'}, inplace=True)

# result_df = pd.concat([F_values_df, start_DF], ignore_index=True)
F_values_df["Tender"] = pd.Categorical(F_values_df["Tender"], categories=tender_order, ordered=True)
F_values_df = F_values_df.sort_values(["Tender","Start", "Finish"])

# df1 = result_df[(result_df.Tender == "Measles") & (result_df.Value == 1)]
df2 = F_values_df

In [34]:
# Transform the 'Value' column into 0s and 1s based on the updated DataFrame
vector1 = df1['Value'].apply(lambda x: 1 if x != 0 else 0).tolist()
vector2 = df2['Value'].apply(lambda x: 1 if x != 0 else 0).tolist()


In [35]:
def dot_product(A, B):
    return sum(a*b for a, b in zip(A, B))

def magnitude(V):
    return sum(v**2 for v in V) ** 0.5

def cosine_similarity(A, B):
    return dot_product(A, B) / (magnitude(A) * magnitude(B))

def cosine_distance(A, B):
    return 1 - cosine_similarity(A, B)


In [36]:
# Calculate cosine distance
distance = cosine_distance(vector1,vector2)
print("Distance: ", distance)
similarity = cosine_similarity(vector1,vector2)
print("Similarity: ", similarity)

Distance:  0.5479538445002807
Similarity:  0.4520461554997192
